In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
cols = [
    "Sex","Length","Diameter","Height","Whole_weight",
    "Shucked_weight","Viscera_weight","Shell_weight","Rings"
]

df = pd.read_csv(r"C:\Users\ACER\Downloads\abalone\abalone.data", names=cols)

df.head()

,Sex,Length,Diameter,Height,Whole_weight,Shucked_weight,Viscera_weight,Shell_weight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4177 entries, 0 to 4176
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Sex             4177 non-null   object 
 1   Length          4177 non-null   float64
 2   Diameter        4177 non-null   float64
 3   Height          4177 non-null   float64
 4   Whole_weight    4177 non-null   float64
 5   Shucked_weight  4177 non-null   float64
 6   Viscera_weight  4177 non-null   float64
 7   Shell_weight    4177 non-null   float64
 8   Rings           4177 non-null   int64  
dtypes: float64(7), int64(1), object(1)
memory usage: 293.8+ KB


In [4]:
df.isnull().sum()

Sex               0
Length            0
Diameter          0
Height            0
Whole_weight      0
Shucked_weight    0
Viscera_weight    0
Shell_weight      0
Rings             0
dtype: int64

In [5]:
df = pd.get_dummies(df, columns=["Sex"])

df.head()

,Length,Diameter,Height,Whole_weight,Shucked_weight,Viscera_weight,Shell_weight,Rings,Sex_F,Sex_I,Sex_M
0,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15,False,False,True
1,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7,False,False,True
2,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9,True,False,False
3,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10,False,False,True
4,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7,False,True,False


In [6]:
X = df.drop("Rings", axis=1).values
y = df["Rings"].values.reshape(-1,1)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [9]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

In [10]:
np.random.seed(42)

input_size = X_train.shape[1]
hidden_size = 8
output_size = 1

W1 = np.random.randn(input_size, hidden_size)
b1 = np.zeros((1, hidden_size))

W2 = np.random.randn(hidden_size, output_size)
b2 = np.zeros((1, output_size))

In [11]:
def forward(X):
    
    z1 = np.dot(X, W1) + b1
    a1 = sigmoid(z1)
    
    z2 = np.dot(a1, W2) + b2
    a2 = z2   # linear output (regression)
    
    return z1, a1, z2, a2

In [12]:
def mse_loss(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

In [13]:
def backward(X, y, z1, a1, z2, a2, lr):

    global W1, W2, b1, b2
    
    m = X.shape[0]

    # Output layer gradient
    dz2 = (a2 - y)
    
    dW2 = np.dot(a1.T, dz2) / m
    db2 = np.sum(dz2, axis=0, keepdims=True) / m

    # Hidden layer gradient
    da1 = np.dot(dz2, W2.T)
    dz1 = da1 * sigmoid_derivative(a1)

    dW1 = np.dot(X.T, dz1) / m
    db1 = np.sum(dz1, axis=0, keepdims=True) / m

    # Update weights
    W1 -= lr * dW1
    W2 -= lr * dW2
    b1 -= lr * db1
    b2 -= lr * db2

In [14]:
epochs = 1000
lr = 0.01

for epoch in range(epochs):
    
    z1, a1, z2, a2 = forward(X_train)

    loss = mse_loss(y_train, a2)

    backward(X_train, y_train, z1, a1, z2, a2, lr)

    if epoch % 100 == 0:
        print("Epoch:", epoch, "Loss:", loss)

Epoch: 0 Loss: 97.30607028841747
Epoch: 100 Loss: 8.797074223128515
Epoch: 200 Loss: 6.810757995463024
Epoch: 300 Loss: 6.383918676077865
Epoch: 400 Loss: 6.134799913177032
Epoch: 500 Loss: 5.927808416354472
Epoch: 600 Loss: 5.737278746892806
Epoch: 700 Loss: 5.566945485642555
Epoch: 800 Loss: 5.42204041050366
Epoch: 900 Loss: 5.301649094861673


In [15]:
_, _, _, y_pred = forward(X_test)

y_pred[:10]

array([[12.01251409],
       [10.6964672 ],
       [13.55513115],
       [12.04420715],
       [10.56204243],
       [10.84066709],
       [10.06020706],
       [ 8.78932139],
       [ 6.6737117 ],
       [10.74398247]])

In [16]:
from sklearn.metrics import mean_squared_error

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("RMSE:", rmse)

RMSE: 2.365512929433422


In [17]:
age_pred = y_pred + 1.5

age_pred[:10]

array([[13.51251409],
       [12.1964672 ],
       [15.05513115],
       [13.54420715],
       [12.06204243],
       [12.34066709],
       [11.56020706],
       [10.28932139],
       [ 8.1737117 ],
       [12.24398247]])